# Paddle Tennis Bandeja Shot Quality Prediction

This notebook demonstrates how to use the trained **Bidirectional LSTM** model to predict the quality grade of a paddle tennis _bandeja_ shot from its coordinate data (JSON file).

## What you need

- A **JSON coordinate file** generated by the pose estimation pipeline. The JSON must contain pelvis, left shoulder, right elbow, and right wrist coordinates extracted from a video clip.

## Don't have a JSON file?

If you only have a video clip (`.mp4`), you need to first extract the coordinates using the pose estimation pipeline:

1. Clone the repository: `git clone https://github.com/CesarEmilioC/thesisRepo_A00830006.git`
2. Follow the **Environment Setup** instructions in the [README](https://github.com/CesarEmilioC/thesisRepo_A00830006/blob/main/README.md)
3. Run pose estimation on your video:
   ```bash
   cd Source
   python main.py pose --video "path/to/your/video.mp4"
   ```
4. The generated JSON will be saved in the `Coordinates/` folder. Use that file with this notebook.

## Classification

The model classifies shots into 5 quality levels:

| Class | Label | Grade Range |
|-------|-------|-------------|
| 0 | Very Low | 1-2 |
| 1 | Low | 3-4 |
| 2 | Medium | 5-6 |
| 3 | High | 7-8 |
| 4 | Excellent | 9-10 |

---

## Step 1: Install Dependencies

Install the required libraries for loading and running the model.

In [ ]:
# Install TensorFlow with Keras 2 compatibility (model was trained with Keras 2.x)
!pip install tf_keras tensorflow numpy -q
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

## Step 2: Select and Load a Trained LSTM Model

Three trained BiLSTMs are available in the repository:

| `MODEL_KEY` | Model | Test accuracy | Spearman ρ |
|-------------|-------|---------------|------------|
| `"final"` *(default)* | `Test03` — the model the thesis discusses | 29.9% | 0.03 |
| `"best_grid"`  | Sensitivity-grid winner: cross-entropy + SGD(0.9) | 35.0% | 0.19 |
| `"best_split"` | Multi-split winner: `train_frac = 0.6`             | 36.1% | 0.23 (p = 0.001) |

The default (`"final"`) reproduces the thesis result and uses the canonical 80/20 stratified split. The other two come from the BiLSTM sensitivity analyses (`Source/Results/SensitivityAnalysis/`, `Source/Results/SplitAnalysis/`). Note that **`best_split` was trained on a 60/40 split**, so its higher accuracy is a sensitivity-study observation rather than a direct apples-to-apples improvement over `Test03`.

Change `MODEL_KEY` in the code cell below to swap models. The selected file is downloaded once from GitHub and cached locally.

In [ ]:
import numpy as np
import json
import urllib.request
import os
import tf_keras as keras
from tf_keras.models import load_model
from tf_keras.preprocessing.sequence import pad_sequences

# ============================================================
# CONFIGURATION - Same constants used in the training pipeline
# ============================================================
MAX_SEQUENCE_LENGTH = 90
NORMALIZATION_EPSILON = 1e-8
CLASS_LABELS = ["Very Low", "Low", "Medium", "High", "Excellent"]

# JSON field names
FIELD_PELVIS = "Pelvis"
FIELD_SHOULDER_RELATIVE = "Left Shoulder Relative"
FIELD_ELBOW_RELATIVE = "Right Elbow Relative"
FIELD_WRIST_RELATIVE = "Right Wrist Relative"

print("Libraries loaded successfully.")

In [ ]:
# ============================================================
# MODEL SELECTOR
# ============================================================
# Change MODEL_KEY to "best_grid" or "best_split" to compare predictions
# from the BiLSTM sensitivity-analysis winners against the thesis baseline.
MODEL_KEY = "final"

MODELS = {
    "final": {
        "name":     "Test03 - BiLSTM Final Model (thesis baseline, 80/20 split)",
        "filename": "lstmModel_Test03_23-03-2026.h5",
        "metrics":  "accuracy 0.299  macro_f1 0.225  spearman 0.03 (p=0.764)",
    },
    "best_grid": {
        "name":     "Sensitivity-grid winner - cross-entropy + SGD(0.9), 80/20 split",
        "filename": "lstmModel_Sens_C_xent_sgd_27-04-2026.h5",
        "metrics":  "accuracy 0.350  macro_f1 0.298  spearman 0.19 (p=0.056)",
    },
    "best_split": {
        "name":     "Multi-split winner - cross-entropy + Adam, 60/40 split",
        "filename": "lstmModel_Split60_27-04-2026.h5",
        "metrics":  "accuracy 0.361  macro_f1 0.278  spearman 0.23 (p=0.001)",
    },
}

selected   = MODELS[MODEL_KEY]
MODEL_PATH = selected["filename"]
MODEL_URL  = f"https://github.com/CesarEmilioC/thesisRepo_A00830006/raw/main/Source/Models/{MODEL_PATH}"

print(f"Active model ({MODEL_KEY!r}): {selected['name']}")
print(f"  Test-set metrics: {selected['metrics']}")

if not os.path.isfile(MODEL_PATH):
    print(f"\nDownloading {MODEL_PATH} from GitHub...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Download complete.")
else:
    print(f"\nUsing cached model: {MODEL_PATH}")

model = load_model(MODEL_PATH)
print(f"\nModel loaded successfully: {MODEL_PATH}")
model.summary()

## Step 3: Define Prediction Functions

These functions replicate the exact same normalization and preprocessing used during training.

In [ ]:
def normalize_sequence(sequence):
    """Per-clip zero-mean, unit-variance normalization.
    Identical to the normalization used during training."""
    mean = np.mean(sequence, axis=0)
    std = np.std(sequence, axis=0) + NORMALIZATION_EPSILON
    return (sequence - mean) / std


def predict_from_json(json_data, model):
    """Predict the quality class from a JSON coordinate dictionary.
    
    Parameters
    ----------
    json_data : dict
        Parsed JSON coordinate data.
    model : keras.Model
        Trained LSTM model.
    
    Returns
    -------
    tuple: (predicted_class, class_label, probabilities)
    """
    pelvis   = np.array(json_data.get(FIELD_PELVIS, []))
    shoulder = np.array(json_data.get(FIELD_SHOULDER_RELATIVE, []))
    elbow    = np.array(json_data.get(FIELD_ELBOW_RELATIVE, []))
    wrist    = np.array(json_data.get(FIELD_WRIST_RELATIVE, []))

    # Build feature sequence (8 features per frame)
    has_shoulder = len(shoulder) > 0
    if has_shoulder:
        L = min(len(pelvis), len(shoulder), len(elbow), len(wrist))
        sequence = np.concatenate(
            [pelvis[:L], shoulder[:L], elbow[:L], wrist[:L]], axis=1
        )
    else:
        L = min(len(pelvis), len(elbow), len(wrist))
        sequence = np.concatenate([pelvis[:L], elbow[:L], wrist[:L]], axis=1)

    # Normalize (same method as training)
    sequence = normalize_sequence(sequence)

    # Pad/truncate to MAX_SEQUENCE_LENGTH
    sequence = pad_sequences(
        [sequence],
        maxlen=MAX_SEQUENCE_LENGTH,
        dtype='float32',
        padding='post',
        truncating='post'
    )

    # Predict
    probabilities = model.predict(sequence, verbose=0)[0]
    predicted_class = np.argmax(probabilities)
    class_label = CLASS_LABELS[predicted_class]

    return predicted_class, class_label, probabilities


print("Prediction functions defined.")

## Step 4: Run a Prediction

### Option A: Use the sample JSON from the repository

We'll download a sample coordinate file directly from the GitHub repository and run a prediction.

In [ ]:
# Download sample JSON from GitHub
SAMPLE_URL = "https://raw.githubusercontent.com/CesarEmilioC/thesisRepo_A00830006/main/Samples/coordinateSamples/player10_part1_clip22_grade9.json"

print("Downloading sample coordinate file...")
response = urllib.request.urlopen(SAMPLE_URL)
sample_data = json.loads(response.read().decode())

# Show metadata
metadata = sample_data.get("metadata", {})
print(f"\nSample file metadata:")
print(f"  Video: {metadata.get('video_name', 'N/A')}")
print(f"  Player: {metadata.get('player_id', 'N/A')}")
print(f"  Grade: {metadata.get('grade', 'N/A')}")
print(f"  Frames: {len(sample_data.get(FIELD_PELVIS, []))}")

# Run prediction
pred_class, label, probs = predict_from_json(sample_data, model)

print(f"\n{'='*50}")
print(f"  PREDICTED CLASS: {pred_class} ({label})")
print(f"{'='*50}")
print(f"\nClass probabilities:")
for i, (lbl, prob) in enumerate(zip(CLASS_LABELS, probs)):
    bar = '#' * int(prob * 40)
    print(f"  {i} ({lbl:>9}): {prob:.4f} {bar}")

### Option B: Upload your own JSON file

Use this cell to upload a JSON coordinate file from your computer and run the prediction on it.

**To use your own file:** Run the cell below, click the "Choose Files" button, and select your JSON file.

In [ ]:
from google.colab import files

print("Upload your JSON coordinate file:")
uploaded = files.upload()

for filename, content in uploaded.items():
    print(f"\nProcessing: {filename}")
    user_data = json.loads(content.decode())

    # Show metadata
    metadata = user_data.get("metadata", {})
    print(f"  Player: {metadata.get('player_id', 'N/A')}")
    print(f"  Grade (if labeled): {metadata.get('grade', 'N/A')}")
    print(f"  Frames: {len(user_data.get(FIELD_PELVIS, []))}")

    # Run prediction
    pred_class, label, probs = predict_from_json(user_data, model)

    print(f"\n{'='*50}")
    print(f"  PREDICTED CLASS: {pred_class} ({label})")
    print(f"{'='*50}")
    print(f"\nClass probabilities:")
    for i, (lbl, prob) in enumerate(zip(CLASS_LABELS, probs)):
        bar = '#' * int(prob * 40)
        print(f"  {i} ({lbl:>9}): {prob:.4f} {bar}")

---

## About this Project

This model is part of the **Paddle Tennis Movement Feedback System** thesis project by Cesar Emilio Castano Marin at Tecnologico de Monterrey.

- **Repository:** [github.com/CesarEmilioC/thesisRepo_A00830006](https://github.com/CesarEmilioC/thesisRepo_A00830006)
- **Default model:** Bidirectional LSTM (`Test03`) trained on 484 clips from 6 players using the canonical 80/20 stratified split.
- **Sensitivity-study models** (`best_grid`, `best_split`): produced by `runSensitivity` and `runSplitAnalysis` in this repository. See README section 14 and `Source/Results/SensitivityAnalysis/`, `Source/Results/SplitAnalysis/` for the full per-run artifacts (learning curves, confusion matrices, classification reports, per-epoch history, summary CSVs).
- **Input:** 4 body keypoints (pelvis, left shoulder, right elbow, right wrist) extracted via OpenPose.
- **Output:** 5-class quality classification of the _bandeja_ shot technique.